# Study 2: Why do agents favor high mitigation?

- Notebook 04 showed frequent high mitigation and little development after the early steps. I want to understand what the agents experience when they make those choices.
- Does development raise income enough to pay for repeated mitigation? Do agents stop developing after they have enough currency, or while currency is still growing?
- How do development gains and pollution penalties contribute to reward at different stages? Does this help explain the preference for high mitigation?- Current-design Q-tables are in `../results/reward_comparison_current`; run Notebook 03 first. Embedded tables below are historical outputs.
- The selected economic reward combines EDI gain with `(income - base income) / 10`. Existing tables and interpretation notes below are historical until the notebook is rerun.

## What does the reward measure?

- A successful development action earns its EDI gain as an immediate reward. Income above the base amount also adds a smaller reward each step, whether the region develops or not.
- Pollution reduces reward every step. Mitigation can improve reward by lowering pollution, even when the region is far from failure.
- Spending currency has no direct reward penalty; it limits which actions the region can afford later. Failed action attempts have a penalty.

## Settings

In [24]:
import json

import pandas as pd

from study_2_economic_sustainability.agents import TabularQLearner
from study_2_economic_sustainability.environment import EconomicSustainabilityEnvironment

In [25]:
REWARD_MODES = ("local", "global", "mixed")
TRAINING_SEEDS = (11, 23, 37, 41, 53)
EVALUATION_SEEDS = tuple(range(100_000, 100_020))
Q_TABLE_DIRECTORY = "../results/reward_comparison_current"

## Load the saved policies

- Each file contains four Q-tables, one for each region.
- I will use the same greedy action choices as notebook 04 and record what happens around each choice.

In [ ]:
agents_by_run = {}

for reward_mode in REWARD_MODES:
    for training_seed in TRAINING_SEEDS:
        filename = f"{Q_TABLE_DIRECTORY}/q_{reward_mode}_{training_seed}.json"
        with open(filename, encoding="utf-8") as file:
            tables = json.load(file)

        agents = []
        for region, table in enumerate(tables):
            agent = TabularQLearner(region)
            for key, values in table.items():
                observation_values = []
                for value in key.split(","):
                    observation_values.append(int(value))
                observation = tuple(observation_values)
                agent.q[observation] = values
            agents.append(agent)

        agents_by_run[(reward_mode, training_seed)] = agents

print(f"Loaded {len(agents_by_run)} trained policies")


## Record what happens at each step

- One row will describe one region at one step. The four regions still act together.
- EDI and currency are recorded before the action. Income, cost, and reward come from that step.

In [27]:
def step_rows(
    step, actions, edi_before, currency_before, pollution_before, rewards, info
):
    rows = []

    for region in range(4):
        capacity_not_used = info["unused_mitigation_capacity"][region]
        rows.append({
            "step": step,
            "region": region,
            "action": actions[region],
            "edi_before": edi_before[region],
            "currency_before": currency_before[region],
            "pollution_before": pollution_before[region],
            "income": info["income"][region],
            "cost_paid": info["costs_paid"][region],
            "mitigation_capacity": info["mitigation_capacity"][region],
            "gross_emissions": info["gross_emissions"][region],
            "emissions_avoided": info["emissions_avoided"][region],
            "remaining_emissions": info["remaining_emissions"][region],
            "capacity_not_used": capacity_not_used,
            "economic_reward": info["economic_rewards"][region],
            "pollution_reward": info["pollution_rewards"][region],
            "received_reward": rewards[region],
        })

    return rows

In [28]:
def collect_episode(agents, reward_mode, seed):
    environment = EconomicSustainabilityEnvironment(reward_mode)
    observations = environment.reset(seed)
    rows = []

    while True:
        edi_before = environment.development_indices
        currency_before = environment.currency_balances
        pollution_before = environment.pollution_loads
        actions = []
        for region in range(4):
            action = agents[region].select_action(observations[region], greedy=True)
            actions.append(action)

        observations, rewards, terminated, truncated, info = environment.step(tuple(actions))
        rows.extend(step_rows(
            environment.step_count, actions, edi_before, currency_before,
            pollution_before, rewards, info
        ))
        if terminated or truncated:
            break

    return rows

In [29]:
step_data = []

for reward_mode in REWARD_MODES:
    for training_seed in TRAINING_SEEDS:
        agents = agents_by_run[(reward_mode, training_seed)]

        for environment_seed in EVALUATION_SEEDS:
            rows = collect_episode(agents, reward_mode, environment_seed)
            for row in rows:
                row["reward_mode"] = reward_mode
                row["training_seed"] = training_seed
            step_data.extend(rows)

steps = pd.DataFrame(step_data)
print(f"Recorded {len(steps):,} region-steps")

Recorded 240,000 region-steps


## What changes over the episode?

- I summarize region-steps across the 20 held-out starts within each training seed, then average the five seed summaries.

In [30]:
steps["stage"] = 1 + (steps["step"] - 1) // 50
steps["develop"] = steps["action"].eq(1)
steps["high_mitigation"] = steps["action"].eq(4)
steps["can_afford_high"] = steps["currency_before"].ge(50)
steps["currency_change"] = steps["income"] - steps["cost_paid"]

stage_by_seed = steps.groupby(["reward_mode", "training_seed", "stage"]).agg(
    develop_percent=("develop", "mean"),
    high_percent=("high_mitigation", "mean"),
    mean_edi=("edi_before", "mean"),
    mean_income=("income", "mean"),
    mean_currency=("currency_before", "mean"),
    currency_change_per_step=("currency_change", "mean"),
    can_afford_high_percent=("can_afford_high", "mean"),
    mean_economic_reward=("economic_reward", "mean"),
    mean_pollution_reward=("pollution_reward", "mean"),
    mean_received_reward=("received_reward", "mean"),
)

for column in ("develop_percent", "high_percent", "can_afford_high_percent"):
    stage_by_seed[column] *= 100

stage_summary = stage_by_seed.groupby(["reward_mode", "stage"]).mean()

### Actions and resources

- `mean_currency` is the balance before a decision.
- `currency_change_per_step` is income minus spending; a positive value means balances grow on average.
- `can_afford_high_percent` shows how often a region had at least 50 currency before acting. It does not say whether high mitigation was its best choice.

In [31]:
resource_columns = [
    "develop_percent", "high_percent", "mean_edi", "mean_income",
    "mean_currency", "currency_change_per_step", "can_afford_high_percent",
]
display(stage_summary[resource_columns].round(1))

develop_percent  high_percent  mean_edi  mean_income  \
reward_mode stage                                                         
global      1                 16.8          33.3      42.8         41.4   
            2                  1.1          66.4      58.4         49.2   
            3                  0.0          64.6      58.7         49.3   
            4                  0.0          64.5      58.7         49.3   
local       1                 21.5          54.4      55.9         47.9   
            2                  0.6          97.6      71.0         55.5   
            3                  0.0          95.1      71.0         55.5   
            4                  0.0          95.0      71.0         55.5   
mixed       1                 19.4          52.5      53.9         46.9   
            2                  0.3          96.2      68.6         54.3   
            3                  0.0          94.5      68.6         54.3   
            4                  0.0          94.1      68.6         54.3   

                   mean_currency  currency_change_per_step  \
reward_mode stage                                            
global      1              415.0                      -0.2   
            2              787.8                       9.5   
            3             1299.3                      10.7   
            4             1834.9                      10.7   
local       1              275.8                      -4.9   
            2              467.6                       5.9   
            3              787.0                       6.7   
            4             1125.6                       6.8   
mixed       1              301.7                      -4.1   
            2              496.4                       5.2   
            3              773.8                       5.9   
            4             1073.4                       6.0   

                   can_afford_high_percent  
reward_mode stage                           
global      1                        100.0  
            2                        100.0  
            3                        100.0  
            4                        100.0  
local       1                         99.9  
            2                        100.0  
            3                        100.0  
            4                        100.0  
mixed       1                         99.9  
            2                        100.0  
            3                        100.0  
            4                        100.0

- Development is mostly an early choice. In steps 1-50, agents develop in 16.8% of global, 21.5% of local, and 19.4% of mixed region-steps. By steps 51-100, those shares fall to 1.1%, 0.6%, and 0.3%. EDI and income then stay almost flat.
- High mitigation moves the other way: about 33% of global actions and 53-54% of local and mixed actions in the first stage, rising to about 65% and 94-98% later.
- Could currency explain this switch? High mitigation costs 50, but regions can afford it on essentially every recorded step, even in the first stage. Balances keep growing after development stops: by steps 151-200, mean currency reaches about 1,835 for global, 1,126 for local, and 1,073 for mixed.
- Income also covers much of the later spending. Local and mixed agents earn about 54-56 currency per step and gain about 6 after costs; global agents earn about 49 and gain about 11 because they choose a broader action mix. I wonder whether development has become less attractive once income can support the preferred mitigation pattern.

### Reward components

- `mean_economic_reward` combines successful EDI gain with the ongoing-income bonus `(income - base income) / 10`. `mean_pollution_reward` is the separate negative pollution term. These are local components before reward pooling.
- `mean_received_reward` is what each agent actually received under its reward mode. It can also include invalid-action and failure penalties.

In [32]:
reward_columns = ["mean_economic_reward", "mean_pollution_reward", "mean_received_reward"]
display(stage_summary[reward_columns].round(2))

mean_edi_gain  mean_pollution_reward  mean_received_reward
reward_mode stage                                                            
global      1               1.13                  -0.74                  0.36
            2               0.04                  -0.15                 -0.12
            3               0.00                  -0.09                 -0.09
            4               0.00                  -0.09                 -0.09
local       1               1.41                  -0.64                  0.69
            2               0.01                  -0.02                 -0.03
            3               0.00                   0.00                  0.00
            4               0.00                   0.00                  0.00
mixed       1               1.36                  -0.60                  0.74
            2               0.01                  -0.01                  0.01
            3               0.00                  -0.00                 -0.00
            4               0.00                  -0.00                 -0.00

- The positive development term is largest in steps 1-50: 1.13 per region-step for global, 1.41 for local, and 1.36 for mixed. It then approaches zero as development actions nearly disappear.
- The pollution term is negative in the first stage, then becomes small under local and mixed policies. Global retains a larger pollution penalty, about -0.09 per region-step in the last two stages. That fits the higher mitigation share and lower pollution seen for local and mixed policies.
- High mitigation has no direct reward bonus. It can help by reducing current and future pollution penalties, while its currency cost is not subtracted from reward. That could make repeated high mitigation attractive once agents can afford it. But these rows show rewards for the actions actually taken, so they cannot tell us how much better high mitigation was than medium or development in the same state.

## Is high mitigation capacity being used?

- High mitigation buys 30 units of capacity for 50 currency. Medium buys 15 units for 25 currency.
- Mitigation only avoids new emissions. It does not remove pollution that has accumulated from earlier steps.
- If a paid high action uses no more than 15 units to avoid new emissions, medium would have produced the same pollution and immediate reward on that step. The saved 25 currency could change later choices, so this is a one-step comparison.

In [33]:
from study_2_economic_sustainability.config import EnvironmentConfig

config = EnvironmentConfig()
medium_capacity = config.mitigation_capacity[3]
high_cost = config.action_costs[4]

paid_high = steps[(steps["action"] == 4) & (steps["cost_paid"] == high_cost)].copy()
paid_high["capacity_used"] = (
    paid_high["mitigation_capacity"] - paid_high["capacity_not_used"]
)
paid_high["medium_would_match"] = paid_high["capacity_used"].le(medium_capacity)

- Each row below covers paid high-mitigation purchases in one 50-step stage. `purchases` is the average count per training seed across its 20 held-out starts.
- `medium_would_match_percent` is the share where medium would have had the same immediate pollution and reward effect.

In [34]:
high_by_seed = paid_high.groupby(["reward_mode", "training_seed", "stage"]).agg(
    purchases=("action", "size"),
    mean_pollution_before=("pollution_before", "mean"),
    mean_capacity_used=("capacity_used", "mean"),
    mean_capacity_not_used=("capacity_not_used", "mean"),
    medium_would_match_percent=("medium_would_match", "mean"),
)
high_by_seed["medium_would_match_percent"] *= 100

high_summary = high_by_seed.groupby(["reward_mode", "stage"]).mean()
display(high_summary.round(1))

purchases  mean_pollution_before  mean_capacity_used  \
reward_mode stage                                                         
global      1         1332.4                    7.2                16.8   
            2         2654.2                    1.3                14.3   
            3         2582.6                    0.9                14.1   
            4         2580.0                    0.9                14.1   
local       1         2176.8                    5.2                16.3   
            2         3902.6                    0.2                13.8   
            3         3805.4                    0.0                13.7   
            4         3800.0                    0.0                13.7   
mixed       1         2099.0                    4.5                15.9   
            2         3849.6                    0.2                13.6   
            3         3778.4                    0.1                13.6   
            4         3765.4                    0.1                13.6   

                   mean_capacity_not_used  medium_would_match_percent  
reward_mode stage                                                      
global      1                        13.2                        63.7  
            2                        15.7                        86.1  
            3                        15.9                        87.5  
            4                        15.9                        87.5  
local       1                        13.7                        67.3  
            2                        16.2                        95.7  
            3                        16.3                        96.5  
            4                        16.3                        96.5  
mixed       1                        14.1                        71.7  
            2                        16.4                        96.6  
            3                        16.4                        97.3  
            4                        16.4                        97.3

- This seems like a real spending question. In steps 101-200, a paid high action uses about 14 of its 30 capacity units under every reward mode. Around 16 units have no effect on that step.
- Medium would have matched high's immediate pollution and reward on 87.5% of late global purchases, 96.5% of late local purchases, and 97.3% of late mixed purchases, while saving 25 currency each time. Even in steps 1-50, it would have matched about 64-72% of high purchases.
- The pollution load before high mitigation is already near zero in later local and mixed steps. I wonder whether agents are buying high capacity mostly to cancel new emissions, even when medium would do that job.
- This makes me question the reward and action design: unused capacity has no direct reward benefit, but the extra spending also has no direct reward penalty. The table does not show why Q-learning prefers high. Coarse observations, learned Q-values, and future effects may matter. Notebook 06 follows up by replacing high with medium across full episodes, without retraining.